In [ ]:
%reload_ext autoreload
%autoreload 2
from importlib import reload

import os
import sys
import pickle
import logging
import warnings
import numpy as np
import astropy as ap
import scipy as sp
import scipy.stats
import matplotlib as mpl
import matplotlib.pyplot as plt

import h5py
import tqdm.notebook as tqdm

import kalepy as kale
import kalepy.utils
import kalepy.plot

import holodeck as holo
import holodeck.sams
import holodeck.gravwaves
from holodeck import cosmo, utils, plot, discrete, sams, host_relations, _PATH_DATA
from holodeck.constants import MSOL, PC, YR, MPC, GYR, SPLC
from pathlib import Path

# Silence annoying numpy errors
np.seterr(divide='ignore', invalid='ignore', over='ignore')
warnings.filterwarnings("ignore", category=UserWarning)

# Plotting settings
mpl.rc('font', **{'family': 'serif', 'sans-serif': ['Times'], 'size': 15})
mpl.rc('lines', solid_capstyle='round')
mpl.rc('mathtext', fontset='cm')
plt.rcParams.update({'grid.alpha': 0.5})
mpl.style.use('default')   # avoid dark backgrounds from dark theme vscode

log = holo.log
log.setLevel(logging.INFO)


# ---- Define filepath containing simulation galaxy merger data files ----#
# ---- (if using files not in _PATH_DATA) ---- #
_HOME_PATH = Path('~/').expanduser()
p = os.path.join(_HOME_PATH, 'cosmo_sim_merger_data')
if os.path.exists(p):
    _SIM_MERGER_PATH = p
else:
    p = os.path.join(_HOME_PATH, 'nanograv/cosmo_sim_merger_data')
    if os.path.exists(p):
        _SIM_MERGER_PATH = p
    else:
        _SIM_MERGER_PATH = _PATH_DATA
print(f"{_SIM_MERGER_PATH=}")
# ------------------------------------------------------------------------ #


In [ ]:
# ---- EDIT PARAMS AS NEEDED TO MATCH
NLOUD = 5
NREALS = 500
NFREQS = 40 
#TAU = 1.0 # fixed hardening timescale [Gyr]
TAU = 2.0 # fixed hardening timescale [Gyr]
NFREQS = 40

# ---- Unpickle SAM data
gpf_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_gpf.pkl'
with open(gpf_pkl_fname, "rb") as f:
    print("unpickling GPF SAM data")
    gpf_data = pickle.load(f)
    sam_gpf, hard_gpf, gwb_new_sam_gpf, gwb_sam_gpf, freqs_gpf, freqs_edges_gpf = gpf_data

gmr_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_gmr.pkl'
with open(gmr_pkl_fname, "rb") as f:
    print("unpickling GMR SAM data")
    gmr_data = pickle.load(f)
    sam_gmr, hard_gmr, gwb_new_sam_gmr, gwb_sam_gmr, freqs_gmr, freqs_edges_gmr = gmr_data


In [ ]:
def gwb_amps(sams=None, dpops=None, fname='compare_gwb_amps', color_hardmods=False, gpf_flags=None):
    
    fig, axs = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=[10,10])

    freqs_to_plot = np.array([1/(10*YR), 1/(3*YR), 1/YR])
    
    for i,f in enumerate(freqs_to_plot):
 
        if i==freqs_to_plot.size-1:
            axs[i].set(xlabel=r'$\log_{10}(A_\mathrm{yr})$')
        axs[i].set(ylabel='Probability Density')
        axs[i].grid(alpha=0.2)
        
        if sams is not None:
            for k,s in enumerate(sams):
                freqs_sam = s.PARS['freqs']
                idx_sam = np.where(np.abs(freqs_sam-f)==np.abs(freqs_sam-f).min())[0]
                hctot = calc_hctot(s.gwb_sam)
                amp_sam = hctot[idx_sam,:].flatten()
                
                lw = 1.5
                col = 'k'
                ls = '-'
                if s.model_type == 'old':
                    lw=1.5
                    col='k'
                    ls='-'
                elif s.model_type == 'old_rc100':
                    lw=2.5
                    col='k'
                    ls=':' if dpops is None else '-'
                elif s.model_type == 'ph15':
                    lw=2.5 if dpops is None else 2
                    col='darkred' if dpops is None else 'k'
                    ls='-.'
                elif s.model_type == 'old_2s':
                    lw=2
                    col='k'
                    ls='--'
                elif s.model_type == 'astr_rc100':
                    lw=2.5 if dpops is None else 3
                    col='r' if dpops is None else 'k'
                    ls='-'
                elif s.model_type == 'astr_nuo0':
                    lw=2
                    col='darkgray'
                    ls='-.'
                elif s.model_type == 'astr':
                    lw=2
                    col='darkgray'
                    ls='-' if dpops is None else ':'
                else:
                    raise ValueError(f"havent defined plot style for {s.model_type=}")

                lbl = s.model_type
                gpf_lbls = [' (GMR)', ' (GPF)']
                if len(gpf_flags) == len(sams):
                    lbl += gpf_lbls[gpf_flags[k]] 
                    
                kale.dist1d(np.log10(amp_sam), density=True, hist=False, confidence=False, carpet=False, 
                            lw=lw, ls=ls, color=col, label=lbl, ax=axs[i])

        if dpops is not None:
            for d in dpops: 
                if sams is not None:
                    if np.any(d.freqs!=freqs_sam):
                        print(f"WARNING: SAM and sim have different frequency arrays.")
                        idx_sim = np.where(np.abs(d.freqs-f)==np.abs(d.freqs-f).min())[0]
                    else:
                        idx_sim = idx_sam
                else:
                    idx_sim = np.where(np.abs(d.freqs-f)==np.abs(d.freqs-f).min())[0]

                amp_sim = d.gwb.strain[idx_sim,:].flatten()  #`strain` is equivalent to `both`
        
                if np.any(~np.isfinite(amp_sim)):
                    print(f"skipping {d.lbl} ({amp_sim.min()}, {amp_sim.max()})")
                else:
                    #print(f"{d.lbl} ({amp_sim.min()}, {amp_sim.max()})")
                    #ls='--' if 'bh0' in d.lbl else '-'
                    print(f"{d.hard_model_type=}")
                    if color_hardmods:
                        col='g' if d.hard_model_type=='old_rc100' else d.color
                        col='k' if d.hard_model_type=='old_rc10' else d.color
                        col='b' if d.hard_model_type=='ph15' else d.color
                        col='m' if d.hard_model_type=='ph15_rc10' else d.color
                    elif not color_hardmods and 'bh0' in d.lbl:
                        col = 'c' 
                    else: 
                        col = d.color
                    #ls = '-' if d.hard_model_type=='old_rc100' else ':'
                    ls = '-' if d.hard_model_type=='ph15' else ':'
                    if 'rTNG300' in d.lbl:
                        ls = '--'
                    else:
                        ls = '-' if d.hard_model_type=='ph15' else ':'
                    kale.dist1d(np.log10(amp_sim), density=True, hist=False, confidence=False, 
                                carpet=False, label='_'.join((d.lbl,d.hard_model_type)), lw=d.lw, 
                                ls=ls, color=col, alpha=0.5, ax=axs[i])
                
        axs[i].set(title=f"Comparison of GWB amplitudes at f={f*YR:.1g} [1/YR]")
        axs[i].legend(fontsize=9)

    plt.suptitle(fname)
    fig.savefig(f'{fname}_nloud{NLOUD}_nreals{NREALS}.png')
    #plt.show()

In [ ]:
def __plot_gwb(fobs, gwb, hc_ss=None, bglabel=None, sslabel=None, **kwargs):
    xx = fobs * YR
    fig, ax = figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN
    )
    if(hc_ss is not None):
        draw_ss_and_gwb(ax, xx, hc_ss, gwb, sslabel=sslabel,
                        bglabel=bglabel, **kwargs)
    else:
        draw_gwb(ax, xx, gwb, **kwargs)
    _twin_hz(ax)
    return fig

def __draw_gwb(ax, xx, gwb, nsamp=10, color=None, label=None, alpha=0.25, **kwargs):
    if color is None:
        color = ax._get_lines.get_next_color()

    kw_plot = kwargs.pop('plot', {})
    kw_plot.setdefault('color', color)
    hh = __draw_med_conf(ax, xx, gwb, plot=kw_plot, label=label, **kwargs)
    if (nsamp is not None) and (nsamp > 0):
        nsamp_max = gwb.shape[1]
        idx = np.random.choice(nsamp_max, np.min([nsamp, nsamp_max]), replace=False)
        for ii in idx:
            ax.plot(xx, gwb[:, ii], color=color, alpha=alpha, lw=1.0, ls='-')

    return hh

def __draw_med_conf(ax, xx, vals, fracs=[0.50, 0.90], weights=None, plot={}, 
                    fill={}, filter=False, label=None, lw=1.0, ls='-'):
    #plot.setdefault('alpha', 0.75)
    #fill.setdefault('alpha', 0.2)
    plot.setdefault('alpha', 0.75)
    fill.setdefault('alpha', 0.1)
    percs = np.atleast_1d(fracs)
    assert np.all((0.0 <= percs) & (percs <= 1.0))

    # center the target percentages into pairs around 50%, e.g.  68 ==> [16,84]
    inter_percs = [[0.5-pp/2, 0.5+pp/2] for pp in percs]
    # Add the median value (50%)
    inter_percs = [0.5, ] + np.concatenate(inter_percs).tolist()
    # Get percentiles; they go along the last axis
    if filter:
        rv = [
            kale.utils.quantiles(vv[vv > 0.0], percs=inter_percs, weights=weights)
            for vv in vals
        ]
        rv = np.asarray(rv)
    else:
        rv = kale.utils.quantiles(vals, percs=inter_percs, weights=weights, axis=-1)

    med, *conf = rv.T
    # plot median
    hh, = ax.plot(xx, med, **plot, lw=lw, ls=ls, label=label)

    # Reshape confidence intervals to nice plotting shape
    # 2*P, X ==> (P, 2, X)
    conf = np.array(conf).reshape(len(percs), 2, xx.size)

    kw = dict(color=hh.get_color())
    kw.update(fill)
    fill = kw

    # plot each confidence interval
    for lo, hi in conf:
        gg = ax.fill_between(xx, lo, hi, **fill)

    return (hh, gg)

def compare_gwb_sim_vs_sam(sams, dpops, gpf_flags=None, 
                           save=True, fname_extra='',sam_colors=None,sam_lbls=None):

    LABEL_GW_FREQUENCY_YR = r"GW Frequency $[\mathrm{yr}^{-1}]$"
    LABEL_GW_FREQUENCY_HZ = r"GW Frequency $[\mathrm{Hz}]$"
    LABEL_GW_FREQUENCY_NHZ = r"GW Frequency $[\mathrm{nHz}]$"
    LABEL_SEPARATION_PC = r"Binary Separation $[\mathrm{pc}]$"
    LABEL_CHARACTERISTIC_STRAIN = r"GW Characteristic Strain"
    LABEL_HARDENING_TIME = r"Hardening Time $[\mathrm{Gyr}]$"
    LABEL_CLC0 = r"$C_\ell / C_0$"

    fig, ax = plot.figax(
        xlabel=LABEL_GW_FREQUENCY_YR,
        ylabel=LABEL_CHARACTERISTIC_STRAIN
    )

    frac = 0.50

    if len(sams) > 0:
        sam_freqs = sams[0].PARS['freqs']
        xx = sam_freqs * YR
        if sam_colors is None: 
            sam_colors = ['k']*len(sams)
        for k,s in enumerate(sams):
            lbl = s.model_type
            if s.model_type == 'old':
                lw=1.5
                col='k'
                ls='-'
            elif s.model_type == 'old_rc100':
                lw=2.5
                col='k'
                ls=':' if len(dpops)==0 else '-'
            elif s.model_type == 'ph15':
                lw=2.5 if len(dpops)==0 else 2
                col='darkred' if len(dpops)==0 else 'k'
                ls='-.'
            elif s.model_type == 'old_2s':
                lw=2
                col='k'
                ls='--'
            elif s.model_type == 'astr_rc100':
                lw=2.5 if len(dpops)==0 else 3
                col='r' if len(dpops)==0 else 'k'
                ls='-'
            elif s.model_type == 'astr_nuo0':
                lw=2
                col='darkgray'
                ls='-.'
            elif s.model_type == 'astr':
                lw=2
                col='darkgray'
                ls='-' if len(dpops)==0 else ':'
            else:
                raise ValueError(f"havent defined plot style for {s.model_type=}")

            gpf_lbls = [' (GMR)', ' (GPF)']
            if len(gpf_flags) == len(sams):
                lbl += gpf_lbls[gpf_flags[k]] 
                ls = '--' if gpf_flags[k] else '-'
            s_hctot = calc_hctot(s.gwb_sam)
            #__draw_gwb(ax, xx, s_hctot, nsamp=0, color=sam_colors[k], label=lbl,
            __draw_gwb(ax, xx, s_hctot, nsamp=0, color=col, label=lbl,
                       lw=lw, ls=ls, alpha=0.05, fracs=[frac])

    if len(dpops) > 0:
        for j,d in enumerate(dpops):
            xx = d.freqs * YR
            #print(f"{d.lbl=}, {d.gwb.both.shape=}")
            #col = 'b' if 'bh0' in d.lbl else d.color
            col = 'c' if 'bh0' in d.lbl else d.color
            lw = 1+0.5*j if 'bh0' in d.lbl else 1.5+0.25*j
            #ls = ':' if 'bh0' in d.lbl else '-.'
            #if d.hard_model_type == 'old_rc100':
            if d.hard_model_type == 'ph15':
                #ls = '-'
                ls = '--' if 'rTNG300' in d.lbl else '-'
                __draw_gwb(ax, xx, d.gwb.both, nsamp=0, color=col, 
                           label=d.lbl, lw=lw, ls=ls, alpha=0.05, fracs=[frac])
            else:
                #ls = ':'
                ls = '--' if 'rTNG300' in d.lbl else ':'
                __draw_gwb(ax, xx, d.gwb.both, nsamp=0, color=col, 
                           lw=lw, ls=ls, alpha=0.05, fracs=[frac])
                
    plt.legend(loc='lower left',fontsize=9)
    plt.suptitle(fname_extra)
    if save:
        #plt.savefig('gwb_compare_ill_tng100_main.png',dpi=300)
        if fname_extra != '':
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_tau{TAU}_{fname_extra}.png'
        else:
            fname = f'gwb_compare_nloud{NLOUD}_nreals{NREALS}_tau{TAU}.png'            
        plt.savefig(fname, dpi=300)


In [ ]:
def plot_loud_binary_params(freqs, sam_bgpars, sam_sspars, dpop, gpf_flag=0, save=False):
    
    fig, axs = plt.subplots(nrows=2, ncols=3, sharex=True, figsize=[10,6])

    bg_alpha = 0.1
    ss_alpha = 0.3
    nskip = 1
    
    sam_bg_m1, sam_bg_m2 = utils.m1m2_from_mtmr(sam_bgpars[0,:,:], sam_bgpars[1,:,:])
    sam_ss_m1, sam_ss_m2 = utils.m1m2_from_mtmr(sam_sspars[0,:,:,:], sam_sspars[1,:,:,:])
    
    dpop_ss_m1, dpop_ss_m2 = utils.m1m2_from_mtmr(dpop.gwb.sspar[0], dpop.gwb.sspar[1])
    dpop_bg_m1, dpop_bg_m2 = utils.m1m2_from_mtmr(dpop.gwb.bgpar[0], dpop.gwb.bgpar[1])
    print(f"{dpop.gwb.bgpar[0].shape=}")
    
    # 1st column panels
    axs[0,0].set(ylabel='Mtot',xscale='log',yscale='log') #xlabel='frequency', 
    axs[0,0].plot(freqs, sam_bgpars[0,:,::nskip]/MSOL,'gray',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
    axs[0,0].plot(freqs, dpop.gwb.bgpar[0][:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real

    axs[1,0].set(ylabel='q',xlabel='frequency', xscale='log') #,yscale='log') #xlabel='frequency', 
    axs[1,0].plot(freqs, sam_bgpars[1,:,::nskip],'gray',lw=0.2,alpha=bg_alpha); # avg mrat of bg sources for each freq for each real
    axs[1,0].plot(freqs, dpop.gwb.bgpar[1][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real

    # 2nd column panels
    axs[0,1].set(ylabel='M1',xscale='log',yscale='log') #xlabel='frequency', 
    axs[0,1].plot(freqs, sam_bg_m1[:, ::nskip]/MSOL,'gray',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
    axs[0,1].plot(freqs, dpop_bg_m1[:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real

    axs[1,1].set(ylabel='M2',xlabel='frequency', xscale='log',yscale='log') #xlabel='frequency', 
    axs[1,1].plot(freqs, sam_bg_m2[:, ::nskip]/MSOL,'gray',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real
    axs[1,1].plot(freqs, dpop_bg_m2[:, ::nskip]/MSOL,'c',lw=0.2,alpha=bg_alpha); # avg mt of bg sources for each freq for each real

    # 3rd column panels
    axs[0,2].set(ylabel='z_init',xscale='log', ylim=(0,2.5)) #,yscale='log')
    axs[0,2].plot(freqs, sam_bgpars[2,:,::nskip],'gray',lw=0.2,alpha=bg_alpha); # avg mrat of bg sources for each freq for each real
    axs[0,2].plot(freqs, dpop.gwb.bgpar[2][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); # avg mrat of bg sources for each freq for each real

    axs[1,2].set(xlabel='frequency', ylabel='z_final',xscale='log', ylim=(0,2.5)) #,yscale='log')
    axs[1,2].plot(freqs, sam_bgpars[3,:,::nskip],'gray',lw=0.2,alpha=bg_alpha); # avg rzi of bg sources for each freq for each real
    axs[1,2].plot(freqs, dpop.gwb.bgpar[3][:, ::nskip],'c',lw=0.2,alpha=bg_alpha); # avg rzi of bg sources for each freq for each real

    print(dpop.gwb.sspar[3].shape) ##nfreq, nloud, nreals

    for ll in range(NLOUD):
        # 1st column panels
        # mt of 10 loudest sources in each freq bin, avg over nreals
        axs[0,0].plot(freqs, np.mean(sam_sspars[0,:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha)
        axs[0,0].plot(dpop.freqs, np.mean(dpop.gwb.sspar[0][:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha) 

        # mrat of 10 loudest sources in each freq bin, avg over nreals
        axs[1,0].plot(freqs, np.mean(sam_sspars[1,:,:,ll],axis=1), 'k.', alpha=ss_alpha) 
        axs[1,0].plot(dpop.freqs, np.mean(dpop.gwb.sspar[1][:,ll,:],axis=1), 'b.', alpha=ss_alpha)

        # 2nd column panels
        # m1 of 10 loudest sources in each freq bin, avg over nreals
        axs[0,1].plot(freqs, np.mean(sam_ss_m1[:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha)
        axs[0,1].plot(dpop.freqs, np.mean(dpop_ss_m1[:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha) 

        # m2 of 10 loudest sources in each freq bin, avg over nreals
        axs[1,1].plot(freqs, np.mean(sam_ss_m2[:,:,ll],axis=1)/MSOL, 'k.', alpha=ss_alpha) 
        axs[1,1].plot(dpop.freqs, np.mean(dpop_ss_m2[:,ll,:],axis=1)/MSOL, 'b.', alpha=ss_alpha)

        # 3rd column panels
        # rzi of 10 loudest sources in each freq bin, avg over nreals
        axs[0,2].plot(freqs, np.mean(sam_sspars[2,:,:,ll],axis=1), 'k.', alpha=ss_alpha) 
        axs[0,2].plot(dpop.freqs, np.mean(dpop.gwb.sspar[2][:,ll,:],axis=1), 'b.', alpha=ss_alpha) 

        # rzf of 10 loudest sources in each freq bin, avg over nreals
        sam_lbl = 'SAM' if ll==0 else None
        sim_lbl = 'sim' if ll==0 else None
        axs[1,2].plot(freqs, np.mean(sam_sspars[3,:,:,ll],axis=1), 'k.', alpha=ss_alpha, label=sam_lbl)
        axs[1,2].plot(dpop.freqs, np.mean(dpop.gwb.sspar[3][:,ll,:],axis=1), 'b.', alpha=ss_alpha, label=sim_lbl) 

        
    axs[1,2].legend(loc='upper left')
        
    plt_title = dpop.lbl+' versus GPF SAM' if gpf_flag else dpop.lbl+' versus GMR SAM' 
    fig.suptitle(plt_title)
    plt.subplots_adjust(hspace=0.1,wspace=0.3,top=0.92)

    if save: 
        fname_extra = 'gpf' if gpf_flag else 'gmr'
        fig.savefig(f'loud_bin_params_nloud{NLOUD}_{dpop.lbl}_{fname_extra}.png', dpi=300)


In [ ]:
def plot_evo(evo, freqs=None, sepa=None, ax=None, label=None, color=None, **kwargs):
    if (freqs is None) and (sepa is None):
        err = "Either `freqs` or `sepa` must be provided!"
        log.exception(err)
        raise ValueError(err)

    if freqs is not None:
        data = evo.at('fobs', freqs)
        xx = freqs * YR
        xlabel = 'GW Frequency [1/yr]'
    else:
        data = evo.at('sepa', sepa)
        xx = sepa / PC
        xlabel = 'Binary Separation [pc]'

    if ax is None:
        fig, ax = plot.figax(xlabel=xlabel)
    else:
        fig = ax.get_figure()

    def _draw_vals_conf(ax, xx, vals, color=color, label=label):
        if color is None:
            color = ax._get_lines.get_next_color()
        if label is not None:
            ax.set_ylabel(label, color=color)
            ax.tick_params(axis='y', which='both', colors=color)
        # vals = np.percentile(vals, [25, 50, 75], axis=0) / units
        vals = utils.quantiles(vals, [0.25, 0.50, 0.75], axis=0).T
        h1 = ax.fill_between(xx, vals[0], vals[-1], alpha=0.2, color=color)
        h2, = ax.plot(xx, vals[1], alpha=0.5, lw=2.0, color=color)
        return (h1, h2)

    # handles = []
    # labels = []

    name = 'Hardening Time [yr]'
    vals = np.fabs(data['sepa'] / data['dadt']) / YR
    _draw_vals_conf(ax, xx, vals, label=name)
    # handles.append(hh)
    # labels.append(name)

    # name = 'eccen'
    # tw = ax.twinx()
    # hh, nn = _draw_vals_conf(tw, freqs*YR, name, 'green')
    # if hh is not None:
    #     handles.append(hh)
    #     labels.append(nn)

    # ax.legend(handles, labels)
    return ax

# load SAMs

In [ ]:
def calc_hctot(gwb_sam):
    return np.sqrt( np.sum(gwb_sam[0]**2,axis=2) + gwb_sam[1]**2 )

def load_sam_data(nloud=1, nreals=10, nfreqs=40, gpf_flag=False, tau=1.0,
                  data_dir=_SIM_MERGER_PATH, fname_type='manual_moddefs'):

    if fname_type=='manual_moddefs':
        samtype = 'gpf' if gpf_flag else 'gmr'
    
        # ---- Unpickle SAM data
        ## right now these files are just stored in the same directory as the notebooks
        ## should move to data dir once things have stabilized a bit
        #sam_pkl_fname=f'sam_nfreqs{NFREQS}_nreals{NREALS}_nloud{NLOUD}_tau{TAU}_{samtype}.pkl'
        #sam_pkl_fname=f'sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_model_type{model_type}_{samtype}.pkl'
        sam_pkl_fname=f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_manual_moddefs_{samtype}_tau{tau}.pkl'

    elif fname_type=='old_new_mods_compare':
        sam_pkl_fname=f'test_sam_nfreqs{nfreqs}_nreals{nreals}_nloud{nloud}_old_new_mods_compare_tau5.55.pkl'
    else:
        raise ValueError(f'{fname_type=} not defined.')
    
    
    with open('/'.join((data_dir, sam_pkl_fname)), "rb") as f:
        print(f'unpickling SAM data: {sam_pkl_fname}')
        sams = pickle.load(f)
        #sam, hard, gwb_new_sam, gwb_sam, freqs, freqs_edges = sam_data
        #sam, hard, gwb_sam, MODEL_PARS = sam_data
   
    #return sam, hard, gwb_new_sam, gwb_sam, freqs, freqs_edges
    #return sam, hard, gwb_sam, MODEL_PARS
    return sams
    

In [ ]:

# ---- Load SAMs
sams_gmr_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                              gpf_flag=False, tau=1.0, data_dir=_SIM_MERGER_PATH)
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_gmr_tau1: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)


In [ ]:
sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sams_list = [ s for s in sams_gmr_tau1 #+sams_gmr_tau3
                  if (s.model_type in sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
#gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
#                  if (s.model_type in sam_mod_list and 
#                  s.PARS['hard_time'] in sam_tau_list) ]

for s in gmr_sams_list:
    print(s.model_type)
sams_list = gmr_sams_list 
gpf_flags = [0]*len(gmr_sams_list) 
#sams_list = gmr_sams_list + gpf_sams_list
#gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 

#gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_gpf_tau1', color_hardmods=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau1', save=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau3', save=False)

# ---- GMR TAU=1
gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_tau1', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb_gmr_tau1', save=False)
## ---- GMR TAU=3
#gwb_amps(sams=sams_gmr_tau3, dpops=None, fname='compare_amps_gmr_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gmr_tau3, [], gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='compare_gwb_gmr_tau3', save=False)
## ---- GPF TAU=1
#gwb_amps(sams=sams_gpf_tau1, dpops=None, fname='compare_amps_gpf_tau1', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau1, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau1', save=False)
## ---- GPF TAU=3
#gwb_amps(sams=sams_gpf_tau3, dpops=None, fname='compare_amps_gpf_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau3, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau3', save=False)


In [ ]:

# ---- Load SAMs
sams_old_new_mods_compare = load_sam_data(nloud=NLOUD, nreals=50, nfreqs=NFREQS, 
                                          gpf_flag=None, tau=None, data_dir=_SIM_MERGER_PATH,
                                          fname_type='old_new_mods_compare')
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_old_new_mods_compare: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)

gpf_flags = [1, 0]

gwb_amps(sams=sams_old_new_mods_compare, dpops=None, fname='compare_amps_sams_old_new_mods_compare', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_old_new_mods_compare, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb__sams_old_new_mods_compare', save=False)


In [ ]:

# ---- Load SAMs
sams_old_new_mods_compare = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
                                          gpf_flag=None, tau=1.0, data_dir=_SIM_MERGER_PATH)
#sams_gmr_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=False, tau=3.0, data_dir=_PATH_DATA)
#sams_gpf_tau1 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=1.0, data_dir=_PATH_DATA)
#sams_gpf_tau3 = load_sam_data(nloud=NLOUD, nreals=NREALS, nfreqs=NFREQS, 
#                              gpf_flag=True, tau=3.0, data_dir=_PATH_DATA)

for s in sams_old_new_mods_compare: #+sams_gmr_tau3+sams_gpf_tau1+sams_gpf_tau3:
    print(s.model_type)

sam_mod_list =['old','old_2s','old_rc100','ph15','astr', 'astr_nuo0','astr_rc100']
sam_mod_list = ['ph15','astr_rc100','astr'] # high, med, low amps
#sam_tau_list = [1.0, 3.0]
sam_tau_list = [1.0]
#sam_tau_list = [3.0]
gmr_sams_list = [ s for s in sams_gmr_tau1 #+sams_gmr_tau3
                  if (s.model_type in sam_mod_list and 
                  s.PARS['hard_time'] in sam_tau_list) ]
#gpf_sams_list = [ s for s in sams_gpf_tau1+sams_gpf_tau3
#                  if (s.model_type in sam_mod_list and 
#                  s.PARS['hard_time'] in sam_tau_list) ]

for s in gmr_sams_list:
    print(s.model_type)
sams_list = gmr_sams_list 
gpf_flags = [0]*len(gmr_sams_list) 
#sams_list = gmr_sams_list + gpf_sams_list
#gpf_flags = [0]*len(gmr_sams_list) + [1]*len(gpf_sams_list) 

#gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_gpf_tau1', color_hardmods=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau1', save=False)
#compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
#                       fname_extra='compare_gwb_gmr_gpf_tau3', save=False)

# ---- GMR TAU=1
gwb_amps(sams=sams_list, dpops=None, fname='compare_amps_gmr_tau1', 
         color_hardmods=False, gpf_flags=gpf_flags)

compare_gwb_sim_vs_sam(sams_list, [], gpf_flags=gpf_flags, 
                       fname_extra='compare_gwb_gmr_tau1', save=False)
## ---- GMR TAU=3
#gwb_amps(sams=sams_gmr_tau3, dpops=None, fname='compare_amps_gmr_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gmr_tau3, [], gpf_flags=[0]*len(sams_gmr_tau1), 
#                       fname_extra='compare_gwb_gmr_tau3', save=False)
## ---- GPF TAU=1
#gwb_amps(sams=sams_gpf_tau1, dpops=None, fname='compare_amps_gpf_tau1', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau1, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau1', save=False)
## ---- GPF TAU=3
#gwb_amps(sams=sams_gpf_tau3, dpops=None, fname='compare_amps_gpf_tau3', color_hardmods=False)

#compare_gwb_sim_vs_sam(sams_gpf_tau3, [], gpf_flags=[1]*len(sams_gpf_tau1), 
#                       fname_extra='compare_gwb_gpf_tau3', save=False)


# OLD STUFF (copied from `compare_gwb_discrete_vs_sam.ipynb`)

In [ ]:
print(len(gwb_sam_gpf))
print(len(gwb_sam_gpf[0]), len(gwb_sam_gpf[1]), 
      len(gwb_sam_gpf[2]), len(gwb_sam_gpf[3])) #, len(gwb_sam_new))
print(gwb_sam_gpf[0].shape) # hcss: [nfreqs, nreals, nloudest]
print(gwb_sam_gpf[1].shape) # hcbg: [nfreqs, nreals]
print(gwb_sam_gpf[2].shape) # sspars: [nsspars, nfreqs, nreals, nloudest]
print(gwb_sam_gpf[3].shape) # bgpars: [nbgpars, nfreqs, nreals]
gwb_sam_gpf_hcss = gwb_sam_gpf[0]
gwb_sam_gpf_hcbg = gwb_sam_gpf[1]
gwb_sam_gpf_hctot = np.sqrt( np.sum(gwb_sam_gpf[0]**2,axis=2) + gwb_sam_gpf[1]**2 )
gwb_sam_gpf_sspars = gwb_sam_gpf[2]
gwb_sam_gpf_bgpars = gwb_sam_gpf[3]

print(gwb_sam_gpf_hcss.shape) # hcss: [nfreqs, nreals, nloudest]
print(gwb_sam_gpf_hcbg.shape) # hcbg: [nfreqs, nreals]
print(gwb_sam_gpf_hctot.shape) # hctot: [nfreqs, nreals]
print(gwb_sam_gpf_sspars.shape) # sspars: [nsspars, nfreqs, nreals, nloudest]
print(gwb_sam_gpf_bgpars.shape) # bgpars: [nbgpars, nfreqs, nreals]



gwb_sam_gmr_hcss = gwb_sam_gmr[0]
gwb_sam_gmr_hcbg = gwb_sam_gmr[1]
gwb_sam_gmr_hctot = np.sqrt( np.sum(gwb_sam_gmr[0]**2,axis=2) + gwb_sam_gmr[1]**2 )
gwb_sam_gmr_sspars = gwb_sam_gmr[2]
gwb_sam_gmr_bgpars = gwb_sam_gmr[3]



# sspars:
# sspar[0,ff,rr,ll] = mt[mm]
# sspar[1,ff,rr,ll] = mr[qq]
# sspar[2,ff,rr,ll] = rz[zz]
# sspar[3,ff,rr,ll] = redz_final[mm,qq,zz,ff]
# bgpars: 
# bgpar[0,ff,rr] = m_bg/sum_bg # bg avg mass
# bgpar[1,ff,rr] = q_bg/sum_bg # bg avg ratio
# bgpar[2,ff,rr] = z_bg/sum_bg # bg avg redshift
# bgpar[3,ff,rr] = zfinal_bg/sum_bg # bg avg redshift after hardening
# bgpar[4,ff,rr] = dcom_bg/sum_bg # bg avg comoving distance after hardening
# bgpar[5,ff,rr] = sepa_bg/sum_bg # bg avg binary separation after hardening
# bgpar[6,ff,rr] = angs_bg/sum_bg # bg avg binary angular separation after hardening


In [ ]:
plot_loud_binary_params(freqs_gpf, gwb_sam_gpf_bgpars, gwb_sam_gpf_sspars, all_fsa_dpops[0], gpf_flag=1, save=True)
#plot_loud_binary_params(freqs_gmr, gwb_sam_gmr_bgpars, gwb_sam_gmr_sspars, all_fsa_dpops[0], save=True)

In [ ]:
compare_gwb_sim_vs_sam(freqs_gpf, [gwb_new_sam_gpf], all_dpops, fsa_dpops=None, outfilename='gwb_compare_all_fid.png')

In [ ]:
fig = holo.plot.plot_gwb(freqs_gpf, gwb_new_sam_gpf)

In [ ]:
fig = holo.plot.plot_gwb(freqs_gpf, gwb_sam_gpf_hctot)

In [ ]:
fig = holo.plot.plot_gwb(freqs_gmr, gwb_new_sam_gmr)

In [ ]:
fig = holo.plot.plot_gwb(freqs_gmr, gwb_sam_gmr_hctot)

In [ ]:
# Calculate the total lifetime of each binary
ncols = 2
nrows = 3
fig, axes = plot.figax(scale='lin', xlabel='Time: actual/specified', ylabel='density',figsize=(8,5),
                       ncols=ncols, nrows=nrows, wspace=0.3,hspace=0.3)
#times = [evo_old_ill.tlook, evo_new_ill.tlook, evo_tng100_1.tlook, evo_tng300_1.tlook, hard.tlook]
times = [dpop_tng100_1.evo.tlook, dpop_tng300_1.evo.tlook] #, hard.tlook]
print(len(times))
i = 0
for idx,ax in np.ndenumerate(axes):
    i = idx[0]*ncols + idx[1]
    if i >= len(times): break
    print(times[i].shape)
    print(f"idx: {idx}, i: {i}")
    dt = times[i][:, 0] - times[i][:, -1]
    # Create figure
    # use kalepy to plot distribution
    kale.dist1d(dt/tau, density=True, ax=ax)
    #ax.legend()
    
plt.show()

In [ ]:
def Mc(M,q):
    return M * q**0.6 / (1+q)**1.2

In [ ]:
#qarr = np.logspace(-5,0,100)
#Marr = np.logspace(4,12,100)
qarr = np.logspace(-2,0,100)
Marr = np.logspace(8,11,100)
#print(qarr)
plt.xscale('log')
plt.yscale('log')
plt.tick_params(axis='y', which='both', left=True, right=True)
plt.plot(qarr,Mc(10**8,qarr))
plt.plot(qarr,Mc(10**9,qarr))
plt.plot(qarr,Mc(10**10,qarr))
plt.plot(qarr,10**10*qarr**0.6)

In [ ]:
1/2**1.2

In [ ]:
plt.xscale('log')
plt.yscale('log')
plt.plot(Marr,Mc(Marr,0.1))
plt.plot(Marr,Mc(Marr,0.5))
plt.plot(Marr,Mc(Marr,1.0))

In [ ]:
plt.xscale('log')
#plt.yscale('log')
plt.plot(Marr,(Mc(Marr,1.0)-Mc(Marr,0.1))/Mc(Marr,0.1))
plt.plot(Marr,(Mc(Marr,1.0)-Mc(Marr,0.01))/Mc(Marr,0.01))
#plt.plot(Marr,Mc(Marr,0.5))
#plt.plot(Marr,Mc(Marr,1.0))

In [ ]:
Mc(1e9,1.0)/Mc(1e9,0.01)

In [ ]:
Mc(1e9,1.0)/Mc(1e9,0.1)

In [ ]:
Mc(1e9,1.0)/Mc(1e8,1.0)

In [ ]:
(Mc(1e10,0.1)-Mc(1e9,0.1))/Mc(1e9,0.1)